# Runtime H₂O ladder — dev screen and confirm

Runs `scripts/run_runtime_h2o.py` on the trained v3 adapter (and optionally the clean base) so the
proxy / physical / ordinary-prefill arms are compared on identical prompts.

**Read `research/campaigns/runtime_h2o_validity/DESIGN_2026_11_19.md` before interpreting anything.**

Order of work:
1. setup + preflight smoke (2 prompts) — fail fast if the adapter, prompt encoding or marker detector is wrong;
2. dev screen (`dev_len`, long bins, 24 prompts, all arms) — decides what enters the confirm, no claims;
3. confirm (`test2` long, already opened once; this is a declared secondary opening on a different estimand);
4. package outputs.

Set `RUN_CONFIRM = True` only after the dev screen looks like a working instrument.

In [ ]:
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

REPO = "https://github.com/kartikbindra/btp-backdoor-research"
COMMIT = "REPLACE_WITH_COMMIT_THAT_ADDS_runtime_h2o"   # must contain src/pfseb/runtime_h2o.py
WORK = Path("/kaggle/working")
SRC = WORK / "btp-research"

gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,uuid,driver_version,compute_cap", "--format=csv,noheader"],
    text=True).strip()
print(gpu)
print(sys.version)
print(platform.platform())
assert "Tesla" in gpu or "NVIDIA" in gpu

if SRC.exists():
    shutil.rmtree(SRC)
subprocess.run(["git", "clone", "--quiet", REPO, str(SRC)], check=True)
subprocess.run(["git", "-C", str(SRC), "checkout", "--quiet", COMMIT], check=True)
head = subprocess.check_output(["git", "-C", str(SRC), "rev-parse", "HEAD"], text=True).strip()
print("repo head", head)
for rel in ("src/pfseb/runtime_h2o.py",
            "scripts/run_runtime_h2o.py",
            "results/campaign_007/v3/qwen15_V3_A_seed42/theta_b.pt",
            "configs/prompts/c007x/prompts.jsonl"):
    p = SRC / rel
    assert p.exists(), f"missing {rel} at {head}"
    print("ok", rel, p.stat().st_size)

import torch, transformers
print("torch", torch.__version__, "transformers", transformers.__version__,
      "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))
(WORK / "session_env.json").write_text(json.dumps({
    "gpu": gpu, "python": sys.version, "platform": platform.platform(),
    "torch": torch.__version__, "transformers": transformers.__version__, "repo_head": head,
}, indent=2))

In [ ]:
def run(tag, arms, pool="dev_len", max_items=0, max_new_tokens=40, max_minutes=0,
        adapter="results/campaign_007/v3/qwen15_V3_A_seed42", model_key="qwen15",
        confirm=False, opening_index=0):
    out = WORK / tag
    cmd = [sys.executable, "-u", "-m", "scripts.run_runtime_h2o",
           "--model_key", model_key, "--pool", pool, "--arms", ",".join(arms),
           "--max_new_tokens", str(max_new_tokens), "--out_dir", str(out),
           "--device", "cuda"]
    if adapter:
        cmd += ["--adapter_dir", adapter]
    if max_items:
        cmd += ["--max_items", str(max_items)]
    if max_minutes:
        cmd += ["--max_minutes", str(max_minutes)]
    if confirm:
        cmd += ["--confirm", "--opening_index", str(opening_index)]
    print("RUN", " ".join(cmd), flush=True)
    started = time.time()
    with (WORK / f"{tag}.log").open("w", buffering=1) as log:
        proc = subprocess.run(cmd, cwd=str(SRC), stdout=log, stderr=subprocess.STDOUT)
    print(tag, "exit", proc.returncode, "minutes", round((time.time() - started) / 60, 1))
    summary = out / "summary.json"
    if summary.exists():
        print(summary.read_text()[:2000])
    return proc.returncode

# Preflight: 2 prompts, two arms, 8 tokens. Catches a wrong adapter, prompt encoding or
# marker detector in ~2 minutes instead of an hour into the dev screen.
run("preflight", ["c0", "proxy_static"], max_items=2, max_new_tokens=8, max_minutes=20)

In [ ]:
DEV_ARMS = ["c0", "proxy_static", "physical_masked", "runtime_masked",
            "runtime_frozen", "runtime_progressive", "runtime_headwise"]
run("dev_screen", DEV_ARMS, pool="dev_len", max_items=24, max_new_tokens=40, max_minutes=180)

In [ ]:
# Confirm. Guarded: flip to True deliberately, after reading the dev screen summary.
RUN_CONFIRM = False
N_CORE = 200      # c0, proxy_static, physical_masked, runtime_masked, runtime_frozen
N_POLICY = 80     # first N of the same id-sorted list, so it is a strict subset
OPENING_INDEX = 2  # secondary opening; record why in the campaign report

if RUN_CONFIRM:
    run("confirm_core", ["c0", "proxy_static", "physical_masked", "runtime_masked",
                         "runtime_frozen"], pool="test2", max_items=N_CORE,
        max_new_tokens=40, max_minutes=600, confirm=True, opening_index=OPENING_INDEX)
    run("confirm_policy", ["c0", "proxy_static", "runtime_progressive", "runtime_headwise"],
        pool="test2", max_items=N_POLICY, max_new_tokens=40, max_minutes=600,
        confirm=True, opening_index=OPENING_INDEX)
else:
    print("confirm not run; set RUN_CONFIRM = True after the dev screen")

In [ ]:
stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
bundle = WORK / f"runtime_h2o_{stamp}"
bundle.mkdir(exist_ok=True)
for item in WORK.iterdir():
    if item.name in {"btp-research", "runtime_h2o_outputs.zip"} or item.name.startswith("runtime_h2o_"):
        continue
    if item.is_file() and (item.suffix in {".json", ".log"} or item.name.endswith(".jsonl")):
        shutil.copy2(item, bundle / item.name)
for tag in ("preflight", "dev_screen", "confirm_core", "confirm_policy"):
    d = WORK / tag
    if d.exists():
        shutil.copytree(d, bundle / tag, dirs_exist_ok=True)
shutil.make_archive(str(WORK / "runtime_h2o_outputs"), "zip", bundle)
print("packaged", bundle)
print(sorted(p.name for p in bundle.iterdir()))